In [13]:
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, IntegerType
from pyspark.sql.functions import max as spark_max
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import * 
from pyspark.sql import Row
from datetime import datetime
import pandas as pd
import requests
import time
import io


StatementMeta(, d531cc17-a328-4cc2-9d5e-dea736e5f40d, 15, Finished, Available, Finished)

In [14]:
spark = SparkSession.builder.appName("Insurance Data Ingestion").getOrCreate()

StatementMeta(, d531cc17-a328-4cc2-9d5e-dea736e5f40d, 16, Finished, Available, Finished)

In [15]:
# if schema does not exists
BRONZE_PATH = "lk_atlas_insurance_data/Tables/dbo"

DATASETS = [
    ("insurance_applicants", "https://raw.githubusercontent.com/inhamo/Datasets-Final/main/insurance_data/insurance_applicants.parquet"),
    ("insurance_policies",  "https://raw.githubusercontent.com/inhamo/Datasets-Final/main/insurance_data/insurance_policies.parquet"),
    ("payment_history",     "https://raw.githubusercontent.com/inhamo/Datasets-Final/main/insurance_data/payment_history.parquet"),
    ("claims_history",      "https://raw.githubusercontent.com/inhamo/Datasets-Final/main/insurance_data/claims_history.parquet"),
]

StatementMeta(, d531cc17-a328-4cc2-9d5e-dea736e5f40d, 17, Finished, Available, Finished)

In [16]:
# -------------------------------------------------
# Initialize watermark table 
# -------------------------------------------------

# Create the watermark dataframe schema
watermark_df = StructType([
    StructField("table_name", StringType(), False), 
    StructField("watermark_value", TimestampType(), False)
])

# Columns which drives the watermark per table 
WATERMARK_COLUMNS = {
    "insurance_applicants": "Effective_Date",
    "insurance_policies": "Effective_Date",
    "claims_history": "Date_of_Claim",
    "payment_history": "Payment_Date"
}

# Create or get watermark table
def initialize_watermark_table():
    """Initialize or get existing watermark table"""
    try:
        df_watermark = spark.table("dbo.watermarktable")
        print("Watermark table already exists")
        return df_watermark
    except:
        print("Creating new watermark table")
        # Insert initial values 
        initial_data = [
            ("insurance_applicants", datetime(2010, 1, 1, 0, 0, 0)),
            ("insurance_policies", datetime(2010, 1, 1, 0, 0, 0)),
            ("payment_history", datetime(2010, 1, 1, 0, 0, 0)),
            ("claims_history", datetime(2010, 1, 1, 0, 0, 0)),
        ]
        
        df_watermark = spark.createDataFrame(initial_data, watermark_df)
        
        # Convert the dataframe to delta table
        (
            df_watermark.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", True)
            .saveAsTable(f"lk_atlas_insurance_data_BRONZE.dbo.watermarktable")
        )
        return df_watermark

# Initialize watermark table
watermark_table = initialize_watermark_table()

# -------------------------------------------------
# Get current watermark for a table
# -------------------------------------------------

def get_current_watermark(table_name):
    """Get the current watermark value for a table"""
    try:
        watermark_df = spark.table("dbo.watermarktable")
        watermark_row = watermark_df.filter(col("table_name") == table_name).collect()
        
        if watermark_row:
            watermark_value = watermark_row[0]["watermark_value"]
            print(f"Current watermark for {table_name}: {watermark_value}")
            return watermark_value
        else:
            print(f"No watermark found for {table_name}, using default")
            return datetime(2010, 1, 1, 0, 0, 0)
    except Exception as e:
        print(f"Error getting watermark for {table_name}: {str(e)[:100]}")
        return datetime(2010, 1, 1, 0, 0, 0)

# -------------------------------------------------
# Process log schema
# -------------------------------------------------

log_schema = StructType([
    StructField("start_time",   TimestampType(), True),
    StructField("end_time",     TimestampType(), True),
    StructField("status",       StringType(),    True),
    StructField("table_name",   StringType(),    True),
    StructField("rows_affected", IntegerType(),   True)
])

# Initialize empty log DataFrame
process_log_df = spark.createDataFrame([], log_schema)

# -------------------------------------------------
# Helper functions
# -------------------------------------------------

def log_event(start_ts, status, table, rows=0):
    """Create a log entry DataFrame"""
    from pyspark.sql import Row
    end_ts = spark.sql("select current_timestamp()").collect()[0][0]
    log_row = Row(start_time=start_ts, end_time=end_ts, status=status, 
                  table_name=table, rows_affected=rows)
    return spark.createDataFrame([log_row], log_schema)

# -------------------------------------------------
# Main ingestion with watermark filtering
# -------------------------------------------------

print("=" * 70)
print("STARTING INSURANCE DATA INGESTION WITH WATERMARK FILTERING")
print("=" * 70)

successful, failed = [], []

for table_name, url in DATASETS:

    print(f"\nProcessing: {table_name}")
    start_ts = spark.sql("select current_timestamp()").collect()[0][0]
    start_time = time.time()

    try:
        # Get current watermark for this table
        current_watermark = get_current_watermark(table_name)
        watermark_column = WATERMARK_COLUMNS.get(table_name)
        
        if not watermark_column:
            print(f"  Warning: No watermark column defined for {table_name}")
            watermark_column = "Effective_Date"  # Default fallback

        # -------------------------
        # Check file availability
        # -------------------------
        print(f"  Checking URL: {url}")
        response = requests.get(url, timeout=30)
        
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code} - File not found")
        
        # Check if content is empty
        content_size = len(response.content)
        print(f"  Content size: {content_size:,} bytes")
        
        if content_size == 0:
            raise Exception(f"Downloaded file is empty (0 bytes)")
        
        # -------------------------
        # Read data and filter by watermark
        # -------------------------
        print("  Reading parquet file...")
        
        # Read into pandas
        df_pandas = pd.read_parquet(io.BytesIO(response.content))
        
        # Convert to Spark DataFrame
        df = spark.createDataFrame(df_pandas)
        
        # Check if watermark column exists
        if watermark_column in df.columns:
            # Filter data to only include records after the watermark
            print(f"  Filtering by watermark: {current_watermark}")
            print(f"  Total rows before filtering: {df.count():,}")
            
            filtered_df = df.filter(col(watermark_column) > current_watermark)
            
            row_count = filtered_df.count()
            col_count = len(filtered_df.columns)
            print(f"  New rows after watermark: {row_count:,} | Columns: {col_count}")
            
            if row_count == 0:
                print("  No new data to process")
                successful.append((table_name, 0, 0))
                continue
        else:
            print(f"  Warning: Watermark column '{watermark_column}' not found in data")
            print(f"  Loading all data")
            filtered_df = df
            row_count = filtered_df.count()
            col_count = len(filtered_df.columns)
            print(f"  Rows: {row_count:,} | Columns: {col_count}")
        
        # -------------------------
        # Write to dbo (append mode for incremental)
        # -------------------------
        try:
            target_table = f"dbo.{table_name}"
            print(f"  Writing to DELTA table: {target_table}")
            
            # Check if table exists
            table_exists = spark.catalog.tableExists(target_table)
            
            if table_exists:
                # Append new data
                filtered_df.write \
                    .format("delta") \
                    .mode("append") \
                    .option("overwriteSchema", True) \
                    .saveAsTable(target_table)
            else:
                # Create table with initial data
                filtered_df.write \
                    .format("delta") \
                    .mode("overwrite") \
                    .option("overwriteSchema", True) \
                    .saveAsTable(target_table)
            
            elapsed = time.time() - start_time
            print(f"  Completed in {elapsed:.2f}s")
            
            successful.append((table_name, row_count, elapsed))
            
            # -------------------------
            # Update watermark
            # -------------------------
            if watermark_column in df.columns and row_count > 0:
                # Get the latest timestamp from the new data
                latest_ts = filtered_df.select(spark_max(watermark_column)).collect()[0][0]
                
                if latest_ts is not None:
                    # Update watermark table
                    from pyspark.sql import Row
                    new_watermark_row = Row(table_name=table_name, watermark_value=latest_ts)
                    new_watermark_df = spark.createDataFrame([new_watermark_row], watermark_df)
                    
                    # Merge into watermark table
                    new_watermark_df.createOrReplaceTempView("new_watermark")
                    
                    spark.sql(f"""
                    MERGE INTO dbo.watermarktable AS tgt
                    USING new_watermark AS src
                    ON tgt.table_name = src.table_name
                    WHEN MATCHED THEN
                      UPDATE SET tgt.watermark_value = src.watermark_value
                    WHEN NOT MATCHED THEN
                      INSERT (table_name, watermark_value)
                      VALUES (src.table_name, src.watermark_value)
                    """)
                    print(f"  Updated watermark to: {latest_ts}")
            
            # -------------------------
            # Log success
            # -------------------------
            process_log_df = process_log_df.union(
                log_event(start_ts, "SUCCESS", table_name, row_count)
            )

        except Exception as e:
            error_msg = str(e)
            print(f"  WRITE FAILED: {error_msg[:200]}")
            
            failed.append((table_name, error_msg))
            
            # Log failure
            process_log_df = process_log_df.union(
                log_event(start_ts, f"FAILED: {error_msg[:100]}", table_name, 0)
            )
            
    except Exception as err: 
        error_msg = str(err)
        print(f"  PROCESSING FAILED: {error_msg[:200]}")
        
        failed.append((table_name, error_msg))
        
        # Log failure
        process_log_df = process_log_df.union(
            log_event(start_ts, f"FAILED: {error_msg[:100]}", table_name, 0)
        )

# -------------------------------------------------
# Summary
# -------------------------------------------------
print("\n" + "=" * 70)
print("INGESTION SUMMARY")
print("=" * 70)

print(f"Successful tables: {len(successful)}")
for t, r, s in successful:
    print(f"  - {t}: {r:,} rows ({s:.2f}s)")

print(f"\nFailed tables: {len(failed)}")
for t, e in failed:
    print(f"  - {t}: {e[:100]}...")

# -------------------------------------------------
# Save process log
# -------------------------------------------------
print("\nSaving process log...")

try:
    if process_log_df.count() > 0:
        process_log_df.write \
            .mode("append") \
            .saveAsTable("dbo.process_log")
        print("Process log saved to: dbo.process_log")
    else:
        print("No process log data to save")
except Exception as e:
    print(f"Error saving process log: {str(e)[:200]}")

print("\n" + "=" * 70)
print("DATA INGESTION COMPLETED")
print("=" * 70)

StatementMeta(, d531cc17-a328-4cc2-9d5e-dea736e5f40d, 18, Finished, Available, Finished)

Creating new watermark table
STARTING INSURANCE DATA INGESTION WITH WATERMARK FILTERING

Processing: insurance_applicants
Current watermark for insurance_applicants: 2010-01-01 00:00:00
  Checking URL: https://raw.githubusercontent.com/inhamo/Datasets-Final/main/insurance_data/insurance_applicants.parquet
  Content size: 586,055 bytes
  Reading parquet file...
  Loading all data
  Rows: 5,912 | Columns: 27
  Writing to DELTA table: dbo.insurance_applicants
  Completed in 13.09s

Processing: insurance_policies
Current watermark for insurance_policies: 2010-01-01 00:00:00
  Checking URL: https://raw.githubusercontent.com/inhamo/Datasets-Final/main/insurance_data/insurance_policies.parquet
  Content size: 1,631,352 bytes
  Reading parquet file...
  Filtering by watermark: 2010-01-01 00:00:00
  Total rows before filtering: 75,313
  New rows after watermark: 75,313 | Columns: 22
  Writing to DELTA table: dbo.insurance_policies
  Completed in 8.59s
  Updated watermark to: 2020-12-31 00:00:00